## Example 1: pydap functionality test 1

In [ ]:
from pydap.model import BaseType, DatasetType
import numpy as np

# Example 1: Basic PyDAP data model creation
try:
    # Create basic data variables
    temperature = BaseType('temperature', data=np.array([20.1, 21.5, 19.8]), 
                          attributes={'units': 'celsius', 'long_name': 'Temperature'})
    
    pressure = BaseType('pressure', data=np.array([1013.2, 1012.8, 1014.1]),
                       attributes={'units': 'hPa', 'long_name': 'Atmospheric Pressure'})
    
    # Create dataset
    dataset = DatasetType('weather_station')
    dataset['temperature'] = temperature
    dataset['pressure'] = pressure
    dataset.attributes = {'title': 'Weather Station Data', 'institution': 'Test Lab'}
    
    print(f"Dataset: {dataset.name}")
    print(f"Variables: {list(dataset.keys())}")
    for var_name in dataset.keys():
        var = dataset[var_name]
        print(f"  {var_name}: {var.data.shape} {var.attributes.get('units', 'no units')}")
    
    print("Example 1 completed successfully")
    
except Exception as e:
    print(f"Error in example 1: {e}")
    raise AssertionError(
        f"pydap example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: pydap functionality test 2

In [ ]:
import requests
from pydap.client import open_url
from pydap.exceptions import ServerError

# Example 2: PyDAP client connection attempt
print("Testing PyDAP client functionality:")

test_servers = {
    "https://test.opendap.org/opendap/data/nc/coads_climatology.nc":
        ['COADSX', 'COADSY', 'TIME', 'SST', 'AIRT', 'UWND', 'VWND'],
    "https://www.ncei.noaa.gov/thredds/dodsC/OisstBase/NetCDF/V2.1/AVHRR/201808/oisst-avhrr-v02r01.20180801.nc":
        ['time', 'anom', 'err', 'ice', 'lat', 'lon', 'sst', 'zlev'],
}

unavailable = {}
for server_url, expected_keys in test_servers.items():
    try:
        print(f"\nAttempting connection to: {server_url[:60]}...")
        dataset = open_url(server_url, timeout=5)
    except (ServerError, requests.exceptions.RequestException) as e:
        # pydap fetches over requests, so an unreachable or refusing server is a RequestException: try the next one
        unavailable[server_url] = f"{type(e).__name__}: {e}"
        print(f"  Server unavailable ({type(e).__name__}); trying the next one")
        continue

    print(f"✓ Connected successfully!")
    print(f"Dataset keys: {list(dataset.keys())[:5]}")  # First 5 keys
    assert list(dataset.keys()) == expected_keys, list(dataset.keys())

    # Get info about the first variable
    first_var = dataset[expected_keys[0]]
    print(f"First variable info: {first_var.name}, shape: {getattr(first_var, 'shape', 'scalar')}")
    assert first_var.name == expected_keys[0] and len(first_var.shape) == 1, (first_var.name, first_var.shape)

    break  # Success, stop trying other servers
else:
    raise AssertionError(f"no OPeNDAP server answered: {unavailable}")

print("\nExample 2 completed - DAP client functionality tested")

## Example 3: pydap functionality test 3

In [ ]:
from pydap.model import BaseType, DatasetType
import numpy as np

# Example 3: Multi-dimensional Data Arrays
try:
    # Create 3D temperature data (time x lat x lon)
    time_data = np.arange(0, 24, 4)
    lat_data = np.array([40.0, 41.0, 42.0])
    lon_data = np.array([-74.0, -73.0, -72.0])
    temp_data = np.random.normal(20.0, 5.0, (6, 3, 3))
    
    # Create dataset with multi-dimensional variable
    dataset = DatasetType('climate_grid')
    dataset['time'] = BaseType('time', data=time_data)
    dataset['lat'] = BaseType('lat', data=lat_data)
    dataset['lon'] = BaseType('lon', data=lon_data)
    dataset['temperature'] = BaseType('temperature', data=temp_data,
                                     attributes={'units': 'celsius'})
    
    print(f"Dataset: {dataset.name}")
    print(f"Temperature shape: {dataset['temperature'].data.shape}")
    print(f"Coordinates: time({len(time_data)}), lat({len(lat_data)}), lon({len(lon_data)})")
    
    print("\nExample 3 completed successfully")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"pydap example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: pydap functionality test 4

In [ ]:
from pydap.model import SequenceType, BaseType, DatasetType
import numpy as np

# Example 4: PyDAP Sequence Type
try:
    # Create a sequence for station data
    station_seq = SequenceType('stations')
    
    # Add variables to sequence
    station_seq['station_id'] = BaseType('station_id')
    station_seq['depth'] = BaseType('depth', attributes={'units': 'm'})
    station_seq['temperature'] = BaseType('temperature', attributes={'units': 'celsius'})
    station_seq['salinity'] = BaseType('salinity', attributes={'units': 'psu'})
    
    # Create dataset with sequence
    dataset = DatasetType('oceanographic_survey')
    dataset['stations'] = station_seq
    
    print(f"Sequence dataset: {dataset.name}")
    print(f"Sequence name: {station_seq.name}")
    print(f"Sequence variables: {list(station_seq.keys())}")
    print(f"Variable count: {len(list(station_seq.keys()))}")
    
    print("\nExample 4 completed successfully")
    
except Exception as e:
    print(f"Error in sequence example: {e}")
    raise AssertionError(
        f"pydap example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: pydap functionality test 5

In [ ]:
from pydap.model import DatasetType, BaseType
import numpy as np

# Example 5: PyDAP Data Slicing and Subsetting
try:
    # Create a 3D temperature dataset
    time_data = np.arange(365)
    lat_data = np.linspace(-90, 90, 181)
    lon_data = np.linspace(-180, 179, 360)
    temp_3d = np.random.normal(15.0, 10.0, (365, 181, 360))
    
    # Create dataset
    dataset = DatasetType('global_climate')
    dataset['time'] = BaseType('time', data=time_data)
    dataset['latitude'] = BaseType('latitude', data=lat_data)
    dataset['longitude'] = BaseType('longitude', data=lon_data)
    dataset['temperature'] = BaseType('temperature', data=temp_3d)
    
    print(f"Dataset: {dataset.name}")
    print(f"Temperature shape: {dataset['temperature'].data.shape}")
    
    # Test data slicing
    january = dataset['temperature'].data[0:30, 90, 180]
    print(f"\nJanuary at 30°N, 0°E: mean={january.mean():.2f}°C")
    
    # Subset by region
    tropics = dataset['temperature'].data[:, 60:120, :]
    print(f"Tropical region shape: {tropics.shape}")
    
    print("\nExample 5 completed successfully")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"pydap example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e